# Prescription OCR — CRNN Training
**Runtime → Change runtime type → T4 GPU** before running.

Pipeline:
1. Mount Drive & clone repo
2. Install dependencies
3. Configure Kaggle credentials
4. Download datasets
5. Build unified manifest & split
6. Train CRNN
7. Export TorchScript model (`model.ptl`)

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Projecat'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted →', DRIVE_DIR)

## 2 — Clone / update repo

In [ ]:
import os
REPO_DIR = '/content/Projecat'

# ── Set your GitHub repo URL here ──────────────────────────────────────────
REPO_URL = 'https://github.com/YOUR_USERNAME/Projecat.git'  # <-- change this
# ───────────────────────────────────────────────────────────────────────────

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 3 — Install dependencies

In [ ]:
!pip install -q \
    datasets>=2.14.0 \
    albumentations>=1.4.0 \
    editdistance \
    fuzzywuzzy \
    python-Levenshtein \
    kaggle

# Verify GPU
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

## 4 — Kaggle credentials
Upload your `kaggle.json` (Account → API → Create New Token) or paste the values below.

In [ ]:
import os, json

# Option A — upload kaggle.json via the Files panel, then run:
# !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B — paste credentials directly (do NOT commit this cell with real values)
KAGGLE_USERNAME = ''  # <-- fill in
KAGGLE_KEY      = ''  # <-- fill in

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('Kaggle credentials written.')
else:
    print('⚠ No Kaggle credentials set — Kaggle datasets will be skipped.')

## 5 — Point data dirs to /tmp (fast local SSD)

In [ ]:
# config.py already uses /tmp/ocr_data — nothing to change.
# Verify paths resolve correctly.
import sys; sys.path.insert(0, '/content/Projecat')
from config import RAW_DIR, PROCESSED_DIR, CHECKPOINTS_DIR
print('RAW_DIR      :', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('CHECKPOINTS  :', CHECKPOINTS_DIR)

## 6 — Download datasets

In [ ]:
# Download HuggingFace datasets (always works — no credentials needed)
!python data/download_all.py --hf-only

# Uncomment to also download Kaggle datasets (requires credentials above)
# !python data/download_all.py --kaggle-only

## 7 — Build manifest & split

In [ ]:
!python data/create_unified_manifest.py
!python data/split_data.py

## 8 — Quick dataset sanity check

In [ ]:
import pandas as pd, os
from config import PROCESSED_DIR

for split in ('train', 'val', 'test'):
    path = os.path.join(PROCESSED_DIR, f'{split}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'{split:5}: {len(df):>7,} rows | sources: {df["source_dataset"].value_counts().to_dict()}')
    else:
        print(f'{split}: NOT FOUND')

## 9 — Train
Adjust `--epochs` and `--batch-size` as needed. T4 comfortably handles batch 64.

In [ ]:
!python model/train.py \
    --epochs 100 \
    --batch-size 256 \
    --lr 1e-3

# To resume from a previous checkpoint:
# !python model/train.py --resume models/checkpoints/best_model.pt --epochs 100

## 10 — Export TorchScript (`model.ptl`)

In [ ]:
import torch, sys
sys.path.insert(0, '/content/Projecat')
from config import CHECKPOINTS_DIR, IMG_HEIGHT, IMG_WIDTH
from model.crnn import CRNN

CKPT = os.path.join(CHECKPOINTS_DIR, 'best_model.pt')
OUT  = '/content/Projecat/models/model.ptl'

model = CRNN()
ckpt  = torch.load(CKPT, map_location='cpu')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

example = torch.zeros(1, 1, IMG_HEIGHT, IMG_WIDTH)
traced  = torch.jit.trace(model, example)
traced.save(OUT)
print(f'Saved → {OUT}  ({os.path.getsize(OUT)/1e6:.1f} MB)')

## 11 — Copy artefacts to Drive

In [ ]:
import shutil

os.makedirs(DRIVE_DIR, exist_ok=True)

for src in [
    '/content/Projecat/models/model.ptl',
    os.path.join(CHECKPOINTS_DIR, 'best_model.pt'),
    os.path.join(CHECKPOINTS_DIR, 'final_model.pt'),
]:
    if os.path.exists(src):
        dst = os.path.join(DRIVE_DIR, os.path.basename(src))
        shutil.copy2(src, dst)
        print(f'Copied {os.path.basename(src)} → Drive')
    else:
        print(f'Not found: {src}')